# 03 — Evaluate Metrics on KITTI-C Predictions

For each prediction:
1. Load raw MiDaS `.npy` prediction
2. Load ground-truth depth
3. Apply scale-and-shift alignment (**required** — MiDaS outputs relative inverse depth)
4. Compute Abs Rel, RMSE, δ1, etc.
5. Save results to `outputs/metrics/kittic_results.csv`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
MODEL_TYPE = 'dpt_hybrid_384'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'kitti_c_manifest.csv'
PRED_DIR = PROJECT_ROOT / 'outputs' / 'predictions' / 'kitti_c'
OUT_CSV = PROJECT_ROOT / 'outputs' / 'metrics' / 'kittic_results.csv'

MIN_DEPTH = 1e-3
MAX_DEPTH = 80.0

In [ ]:
import numpy as np
import pandas as pd
import cv2
from tqdm.notebook import tqdm

from src.evaluation.align import align_scale_shift
from src.evaluation.metrics import compute_all_metrics
from src.datasets.transforms import load_kitti_depth
from src.utils.io import save_metrics_csv

df = pd.read_csv(MANIFEST_PATH)
df = df[df['gt_path'].notna()].reset_index(drop=True)
print(f'Evaluating {len(df)} samples')

In [ ]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_path = PRED_DIR / row['corruption_type'] / str(row['severity']) / f"{row['frame_id']}.npy"
    if not pred_path.exists():
        continue  # inference not yet run for this sample

    try:
        pred = np.load(str(pred_path))
        gt = load_kitti_depth(row['gt_path'])

        if gt.shape != pred.shape:
            gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST)

        valid_mask = gt > 0
        aligned = align_scale_shift(pred, gt, valid_mask)
        m = compute_all_metrics(aligned, gt, valid_mask, MIN_DEPTH, MAX_DEPTH)

        records.append({
            'image_path': row['image_path'],
            'gt_path': row['gt_path'],
            'pred_path': str(pred_path),
            'dataset': 'kitti_c',
            'corruption_type': row['corruption_type'],
            'severity': row['severity'],
            'sequence': row['sequence'],
            'frame_id': row['frame_id'],
            'model_name': MODEL_TYPE,
            **m,
        })

    except Exception as e:
        print(f'ERROR: {row["image_path"]}: {e}')

results_df = pd.DataFrame(records)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUT_CSV, index=False)
print(f'Saved {len(results_df)} results to {OUT_CSV}')

In [ ]:
from src.analysis.report_tables import corruption_summary_table

print('=== Mean metrics per corruption type ===')
corruption_summary_table(results_df)

## Next step

Run `04_failure_case_analysis.ipynb` to visualise worst/median/best failures.